In [ ]:
pip install pandas numpy scikit-learn nltk

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

import nltk
import string

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
df = pd.read_csv("/content/spam.csv", encoding='latin-1')

In [ ]:
df = df[['v1', 'v2']]

df.columns = ['label', 'message']

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [ ]:
ps = PorterStemmer()

def preprocess_text(text):

    text = text.lower()

    words = text.split()

    words = [word for word in words if word not in stopwords.words('english')]

    words = [word.strip(string.punctuation) for word in words]

    words = [ps.stem(word) for word in words]

    return " ".join(words)

In [ ]:
df['message'] = df['message'].apply(preprocess_text)

In [ ]:
cv = CountVectorizer()

X = cv.fit_transform(df['message']).toarray()

y = df['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = MultinomialNB()

model.fit(X_train, y_train)

MultinomialNB()

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9820627802690582

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.94      0.93      0.93       150

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
def predict_spam(message):

    processed = preprocess_text(message)

    vector = cv.transform([processed]).toarray()

    prediction = model.predict(vector)

    if prediction[0] == 1:
        return "Spam"
    else:
        return "Not Spam"

In [ ]:
msg1 = "Congratulations! You won a free iPhone. Click now!"

msg2 = "Hey, are we meeting tomorrow?"

print(msg1, "->", predict_spam(msg1))

print(msg2, "->", predict_spam(msg2))

Congratulations! You won a free iPhone. Click now! -> Spam
Hey, are we meeting tomorrow? -> Not Spam


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(df['message'])